In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.metrics import precision_recall_curve
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("train_u6lujuX_CVtuZ9i.csv")

In [ ]:
df.shape
df.info()
df.isnull().sum()

In [ ]:
df["Gender"] = df["Gender"].fillna(df["Gender"].mode()[0])
df["Married"] = df["Married"].fillna(df["Married"].mode()[0])
df["Dependents"] = df["Dependents"].fillna(df["Dependents"].mode()[0])
df["Self_Employed"] = df["Self_Employed"].fillna(df["Self_Employed"].mode()[0])
df["Loan_Amount_Term"] = df["Loan_Amount_Term"].fillna(df["Loan_Amount_Term"].mode()[0])
df["Credit_History"] = df["Credit_History"].fillna(df["Credit_History"].mode()[0])
df["LoanAmount"] = df["LoanAmount"].fillna(df["LoanAmount"].median())

In [ ]:
df = df.drop("Loan_ID", axis=1)

In [ ]:
df["Gender"] = df["Gender"].map({"Male":1,  "Female":0})
df["Married"] = df["Married"].map({"Yes":1,  "No":0})
df["Education"] = df["Education"].map({"Graduate":1,  "Not Graduate":0})
df["Self_Employed"] = df["Self_Employed"].map({"Yes":1,  "No":0})

df["Dependents"] = df["Dependents"].replace("3+", 3).astype(int)

df = pd.get_dummies(df, columns=["Property_Area"], drop_first=True)

In [ ]:
df["LoanToIncomeRatio"] = df["ApplicantIncome"]/df["LoanAmount"]
df["TotalIncome"] = df["ApplicantIncome"]+df["CoapplicantIncome"]
df["EMI Approx"] = df["LoanAmount"]/df["Loan_Amount_Term"]*100
df["IncomePerDependent"] = df["TotalIncome"]/(df["Dependents"] + 1)

In [ ]:
x = df.drop("Loan_Status", axis=1)
y = df["Loan_Status"].map({"Y":1, "N":0})

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

## Basic Logistic Regression Model

In [ ]:
model_lr = LogisticRegression(max_iter=100000)
model_lr.fit(x_train_scaled, y_train)

y_pred_lr = model_lr.predict(x_test_scaled)

print("Logistic Regression:")
print(classification_report(y_test, y_pred_lr))

## Imbalance Handling

In [ ]:
model_lr_bal = LogisticRegression(class_weight='balanced', max_iter=100000)
model_lr_bal.fit(x_train_scaled, y_train)

y_pred_lr_bal = model_lr_bal.predict(x_test_scaled)

print("Logistic Regression (Balanced):")
print(classification_report(y_test, y_pred_lr_bal))

# Final Model Random Forest

In [ ]:
model_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    class_weight="balanced",
    random_state=42
)

model_rf.fit(x_train, y_train)

y_pred_rf = model_rf.predict(x_test)

print(classification_report(y_test, y_pred_rf))

## Feature Importance

In [ ]:
feature_importances = pd.Series(model_rf.feature_importances_, index=x.columns)
feature_importances.sort_values(ascending=False).head(10).plot(kind="barh")

plt.title("Top Feature Importances")
plt.show()

## Conclusion

### The objective of this project was to predict loan approval using machine learning techniques.

### Initial modeling using logistic regression revealed a bias toward the majority class (approved loans), resulting in poor detection of rejected applications. To address this, class imbalance handling techniques and threshold tuning were applied, but they provided only limited improvement.

### Feature engineering played a crucial role in enhancing model performance. By introducing financially meaningful features such as total income, loan-to-income ratio, and income per person, the model was better able to capture repayment capacity, leading to an improvement in minority class recall from 0.58 to 0.63.

### A Random Forest model was then implemented to capture non-linear relationships, achieving slightly better performance compared to logistic regression. However, overall performance remained constrained, indicating that the primary limitation lies in the dataset’s feature representation rather than model complexity.

### In conclusion, this project demonstrates that domain-driven feature engineering is more impactful than simply applying more complex models and highlights the importance of understanding both data and problem context in machine learning.